In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# A Backtracking Solver for CSPs

In [ ]:
type Variable = string;
type Value =  string | number;
type Formula = string;
type Assignment = Record<Variable, Value>;
type CSP = [Variable[], Value[], Formula[]];
type VisualizationCallback = (assignment: Assignment) => void;

interface AnnotatedConstraint {
    formula: Formula;
    vars: Set<Variable>;
}

The function `collectVariables(expr)` takes a string `expr` that can be interpreted as a valid expression as its input and collects all variables occurring in `expr`.
It takes care to remove names that correspond to predefined values or functions. To achieve this, it defines a set of `builtIns` containing reserved names (like `Math`, `true`, `false`, `abs`, `min`) that are not treated as variables, ensuring that only true variable names are returned by `collectVariables`.

In [ ]:
function collectVariables(expr: string): Set<string> {
    const identifierRegex = /[a-zA-Z_][a-zA-Z0-9_]*/g;
    const builtIns = new Set(['abs', 'min', 'max', 'pow', 'sum', 'len', 'Math', 'true', 'false']);
    const variables = new Set<string>();
    let match: RegExpExecArray | null;
    while ((match = identifierRegex.exec(expr)) !== null) {
        if (!builtIns.has(match[0])) variables.add(match[0]);
    }
    return variables;
}

In [ ]:
const expr = 'abs(x - y) + abs(z1 - z2)';
collectVariables(expr);

## The Backtracking Solver with Animation

The function `isConsistent(variable, value, assignment, constraints)` takes four arguments:
- `variable` is a variable that does not occur in `assignment`,
- `value` is a value that can be substituted for this variable,
- `assignment` is a *consistent* partial variable assignment.
- `constraints` is a set of pairs of the form $\langle f, V \rangle$ where $f$ is a formula and $V$ is the set of variables occurring in $f$.

This function returns `true` iff the partial variable assignment
$$\texttt{assignment} \cup \bigl\{\langle\texttt{variable} \mapsto\texttt{value}\rangle\bigr\}$$
is consistent with all the constraints $f$ occurring in `constraints`.

**Note** that the function `evaluateExpression` handles the variable substitution dynamically.

In [ ]:
function isConsistent(
    variable: Variable, 
    value: Value, 
    assignment: Assignment, 
    constraints: AnnotatedConstraint[]
): boolean {
    const newAssignment = { ...assignment, [variable]: value };
    const assignedVars = new Set(Object.keys(newAssignment));

    for (const { formula, vars } of constraints) {
        if (vars.has(variable) && isSubset(vars, assignedVars)) {
            if (!evaluateExpression(formula, newAssignment)) return false;
        }
    }
    return true;
}
function evaluateExpression(expr: string, context: Assignment): boolean {
    let jsExpr = expr.replace(/\band\b/g, '&&').replace(/\bor\b/g, '||').replace(/\bnot\b/g, '!');
    const argNames = Object.keys(context);
    const argValues = Object.values(context);
    try {
        return new Function(...argNames, `return (${jsExpr});`)(...argValues);
    } catch (e) { return false; }
}
function isSubset(subset: Set<string>, superset: Set<string>): boolean {
    for (let elem of subset) if (!superset.has(elem)) return false;
    return true;
}

The function `backtrackSearch` takes four arguments:
- `assignment` is a partial variable assignment that is represented as a dictionary. Initially, this assignment will be the empty dictionary. Every recursive call of `backtrackSearch` adds the assignment of one variable to the given assignment. The important invariant of recursive calls of `backtrackSearch` is that `assignment` is *consistent*, i.e. all constraints $f$ that contain only variables from the set $\mathtt{dom}(\mathtt{assignment})$ are satisfied.
- `csp` is an *augmented constraint satisfaction problem*,
  i.e. `csp` is a triple of the form
  $$ \mathcal{P} = \langle \mathtt{Vars}, \mathtt{Values}, \mathtt{Constraints} \rangle $$
  where
  - $\mathtt{Vars}$ is a set of strings which serve as *variables*,
  - $\mathtt{Values}$ is a set of *values* that can be assigned
    to the variables in $\mathtt{Vars}$.
  - $\mathtt{Constraints}$ is a set of pairs of the form $(f, V)$ where $f$ is a Boolean expression, while $V$ is the set of variables occurring in $f$.
- `state` is an object tracking the number of steps.
- `onUpdate` (optional) is a callback function for visualization.

The function `backtrackSearch` tries to find a solution of `csp` by recursively augmenting `assignment`.

In [ ]:
function backtrackSearch(
    assignment: Assignment, 
    csp: [Variable[], Value[], AnnotatedConstraint[]], 
    state: { steps: number },
    onUpdate?: VisualizationCallback 
): [number, Assignment] | null {
    if (onUpdate) {
        onUpdate(assignment);
    }
    const [variables, values, constraints] = csp;
    if (Object.keys(assignment).length === variables.length) {
        return [state.steps, assignment];
    }
    const unassignedVar = variables.find(v => !(v in assignment));
    if (!unassignedVar) return null;
    for (const value of values) {
        state.steps++;
        if (isConsistent(unassignedVar, value, assignment, constraints)) {
            const newAssignment = { ...assignment, [unassignedVar]: value };
            const result = backtrackSearch(newAssignment, csp, state, onUpdate);
            if (result) return result;
        }
    }
    return null;
}

The procedure `solve(csp)` takes a *constraint satisfaction problem*
`csp` as input. Here `csp` is a triple of the form
$$ \mathcal{P} = \langle \mathtt{Variables}, \mathtt{Values}, \mathtt{Constraints} \rangle $$
where
- `variables` is a set of strings which serve as *variables*,
- `values` is a set of *values* that can be assigned
  to the variables in the set `variables`.
- `constraints` is a set of formulas (constraints).

The main purpose of the function `solve` is to convert the CSP `csp` into an
*augmented CSP* where every constraint $f$ is annotated with
the variables occurring in $f$. This annotated CSP is then solved using the function
`backtrackSearch`.

In [ ]:
function solve(
    csp: CSP, 
    onUpdate?: VisualizationCallback
): [number, Assignment] | null {
    const [variables, values, constraints] = csp;
    const state = { steps: 0 };
    const annotatedConstraints = constraints.map(f => ({
        formula: f,
        vars: collectVariables(f)
    }));
    const result = backtrackSearch({}, [variables, values, annotatedConstraints], state, onUpdate);
    console.log(`Tested ${state.steps} partial assignments`);
    return result;
}